# GridNet-HD — label transfer ภายใต้ pose error ทุกภาพของ val zones (CPU, ไม่ใช้ GPU)

การทดลองของรูป motivating ใน paper ทำใหม่ให้ครบทั้งโซน: จุดที่กล้องแต่ละภาพมองเห็น รับ label จาก pixel ของ mask (ของผู้สร้าง dataset) ที่มัน project ไปตก
ครั้งแรกด้วย pose ที่ให้มากับ dataset แล้วด้วย pose ที่ถูกรบกวน (ตำแหน่ง 2–50 ซม., มุม 0.05–1°) — ไม่มี network เกี่ยวข้อง
รูปเดิมใน paper มาจาก **ภาพเดียว** ของ t5b5; notebook นี้ทำ **ทุกภาพที่มี mask ของ 6 val zones** (ประมาณ 850 ภาพ) ด้วย error model เดียวกับ protocol (σ ต่อแกน)

**Runtime: CPU + High-RAM** (Runtime → Change runtime type → CPU, High-RAM) — 8 cores ทำงานขนานกัน
เวลาที่วัดได้ (14 พ.ย. 2026, High-RAM 8 cores): 1.75 ชม. สำหรับ 6 โซน 852 ภาพ (12.5–25.8 นาทีต่อโซน); CPU ธรรมดา (2 cores) ช้ากว่า ~4 เท่า → ตั้ง `STRIDE = 3`
ใช้ compute units น้อยมาก (ไม่มี GPU) · หยุดกลางทางได้: โซนที่เสร็จแล้วอยู่บน Drive และถูกข้ามเมื่อรันใหม่

**ก่อนรัน**: `GridNet/code/gridnet_code.zip` บน Drive ต้องเป็นของ **update36 ขึ้นไป** (cell 2 ตรวจให้)
**ผลลัพธ์**: `GridNet/label_transfer/<TAG>/<zone>/label_transfer.csv` ต่อโซน และ `.../all/label_transfer.csv` + รูป รวมทุกโซน — เสร็จแล้วผลอยู่บน Drive

In [ ]:
#@title 1. Mount Drive + parameters
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/GridNet'
CODE_ZIP = f'{DRIVE}/code/gridnet_code.zip'      #@param {type:"string"}
CACHE_DRIVE = f'{DRIVE}/cache'
TAG = 'v1'                                       #@param {type:"string"}
ZONES = 't1z5b,t1z8,t3z4,t4z1,t5b1,t5b5'         #@param {type:"string"}
NOISE = 'protocol'                               #@param ["protocol", "fixed"]
DRAWS = 1                                        #@param {type:"integer"}
STRIDE = 1                                       #@param {type:"integer"}
WORKERS = 0                                      #@param {type:"integer"}
SHUTDOWN_WHEN_DONE = False                       #@param {type:"boolean"}
OUT = f'{DRIVE}/label_transfer/{TAG}'
zones = [z.strip() for z in ZONES.split(',') if z.strip()]
missing = [z for z in zones if not os.path.exists(f'{CACHE_DRIVE}/{z}/points.npz')]
assert not missing, f"no LiDAR cache on Drive for {missing}"
os.makedirs(OUT, exist_ok=True)
cores = os.cpu_count(); ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30
print(f"{cores} CPU cores, {ram:.0f} GB RAM" + ("" if ram > 20 else "   <- standard runtime: slow (2 cores); High-RAM is recommended, or set STRIDE = 3"))
done = [z for z in zones if os.path.exists(f'{OUT}/{z}/label_transfer.json')]
print(f"error model {NOISE}, {DRAWS} draw(s) per image and level, every {STRIDE}. image  ->  {OUT}")
print(f"{len(done)}/{len(zones)} zones already on Drive:", done)


In [ ]:
#@title 2. Code
import subprocess, sys, zipfile, os
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "pillow", "matplotlib"], check=True)
os.makedirs('/content/code', exist_ok=True)
zipfile.ZipFile(CODE_ZIP).extractall('/content/code')
assert os.path.exists('/content/code/scripts/label_transfer_zone.py'), "gridnet_code.zip on Drive is older than update36: pack and upload it again"
r = subprocess.run([sys.executable, "-c", "import sys; sys.path.insert(0,'/content/code'); import gridnet.label_transfer, gridnet.io; print('imports ok')"],
                   capture_output=True, text=True)
print(r.stdout.strip() or r.stderr[-1500:])
assert 'imports ok' in r.stdout


In [ ]:
#@title 3. One zone at a time: points + masks + poses -> label transfer -> Drive (resumable)
import subprocess, sys, os, shutil, time, json
from huggingface_hub import snapshot_download
SCRIPT = '/content/code/scripts/label_transfer_zone.py'
LOCAL = f'/content/label_transfer/{TAG}_{NOISE}_{DRAWS}'      # one local folder per setting: records of another setting are never mixed in

def run(cmd):
    # run the script and show its progress lines; returns the exit code
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=dict(os.environ, PYTHONPATH='/content/code'))
    tail = []
    try:
        for line in p.stdout:
            tail.append(line); del tail[:-25]
            print('   ' + line, end='', flush=True)
    except KeyboardInterrupt:
        p.terminate(); p.wait(); print("stopped by user; finished zones are on Drive -> run the cell again to continue"); raise
    p.wait()
    return p.returncode

def settings_match(zone):
    # a zone on Drive counts as done only when it was run with the settings of cell 1
    path = f'{OUT}/{zone}/label_transfer.json'
    if not os.path.exists(path): return False
    d = json.load(open(path)); s = d['settings']
    full = d['images'] == len(range(0, d['images_with_mask'], STRIDE))
    return s['noise'] == NOISE and s['draws'] == DRAWS and full

t_all = time.time()
for z in zones:
    if settings_match(z):
        print(f"[skip] {z}: on Drive"); continue
    t0 = time.time()
    os.makedirs(f'/content/cache/{z}', exist_ok=True)
    if not os.path.exists(f'/content/cache/{z}/points.npz'):
        shutil.copy(f'{CACHE_DRIVE}/{z}/points.npz', f'/content/cache/{z}/points.npz')
    snapshot_download("heig-vd-geo/GridNet-HD", repo_type="dataset", allow_patterns=[f"{z}/masks/*", f"{z}/pose/*"], local_dir="/content/data")
    n_masks = len(os.listdir(f'/content/data/{z}/masks'))
    print(f"[{z}] points + {n_masks} masks + poses on local disk in {(time.time() - t0) / 60:.1f} min", flush=True)
    code = run([sys.executable, SCRIPT, '--cache', '/content/cache', '--data', '/content/data', '--zone', z, '--out', LOCAL,
                '--noise', NOISE, '--draws', str(DRAWS), '--stride', str(STRIDE), '--workers', str(WORKERS)])
    if code != 0:
        print(f"[fail] {z} (exit {code}) - its finished images stay in {LOCAL}/{z}; run the cell again"); continue
    os.makedirs(f'{OUT}/{z}', exist_ok=True)
    for f in os.listdir(f'{LOCAL}/{z}'):
        shutil.copy(f'{LOCAL}/{z}/{f}', f'{OUT}/{z}/{f}')
    shutil.rmtree(f'/content/data/{z}', ignore_errors=True); os.remove(f'/content/cache/{z}/points.npz')
    print(f"[{z}] done in {(time.time() - t0) / 60:.1f} min  ->  {OUT}/{z}", flush=True)
ok = [z for z in zones if settings_match(z)]
print(f"finished {len(ok)}/{len(zones)} zones in {(time.time() - t_all) / 60:.1f} min" + ("" if len(ok) == len(zones) else "  -> run this cell again"))


In [ ]:
#@title 4. All zones pooled: table + figure -> Drive
import subprocess, sys, os, shutil, json
from IPython.display import Image, display
ok = [z for z in zones if settings_match(z)]
assert ok, "no finished zone yet"
for z in ok:                                    # the pooled result is computed from the copies on Drive
    if not os.path.exists(f'{LOCAL}/{z}/label_transfer.json') or not os.path.exists(f'{LOCAL}/{z}/records.jsonl'):
        shutil.rmtree(f'{LOCAL}/{z}', ignore_errors=True); shutil.copytree(f'{OUT}/{z}', f'{LOCAL}/{z}')
assert run([sys.executable, SCRIPT, '--out', LOCAL, '--combine', ','.join(ok)]) == 0
assert run([sys.executable, '/content/code/scripts/plot_label_transfer.py', '--csv', f'{LOCAL}/all/label_transfer.csv',
            '--out', f'{LOCAL}/all/label_transfer_val']) == 0
os.makedirs(f'{OUT}/all', exist_ok=True)
for f in os.listdir(f'{LOCAL}/all'):
    shutil.copy(f'{LOCAL}/all/{f}', f'{OUT}/all/{f}')
display(Image(f'{LOCAL}/all/label_transfer_val.png', width=900))
d = json.load(open(f'{LOCAL}/all/label_transfer.json'))
print(f"{len(ok)}/{len(zones)} zones, {d['images']} images  ->  {OUT}/all/  (label_transfer.csv, label_transfer.json, label_transfer_val.pdf/.png)")
if len(ok) < len(zones): print("NOT all zones yet:", [z for z in zones if z not in ok], "-> run cell 3 again, then this cell")


In [ ]:
#@title 5. (optional) Shut the runtime down once everything above has finished
if SHUTDOWN_WHEN_DONE:
    from google.colab import drive, runtime
    print("flushing pending writes to Drive ..."); drive.flush_and_unmount()
    print("results are on Drive; shutting the runtime down"); runtime.unassign()
else:
    print("SHUTDOWN_WHEN_DONE is off -> remember to Disconnect and delete runtime")


## หมายเหตุ
- นิยาม (เหมือนการทดลองภาพเดียวเดิม): จุดที่ "มองเห็น" = ผ่าน z-buffer ของกล้องที่ pose ถูกรบกวน (depth map 1/4, tolerance 0.5 ม. เท่ากับขั้น association ของ pipeline);
  ต่อ class = สัดส่วนจุดของ class นั้นที่ pixel ที่ตกมี label เดียวกัน (ตกบน pixel ที่ไม่มี label เช่นสายไฟตกบนท้องฟ้า = พลาด); `all` = ทุกจุดที่ตกบน pixel ที่มี label
- จุดที่ใช้คือจุด voxel 5 ซม. ของ cache (label ข้างมาก) ไม่ใช่จุดดิบของ LAS — ตัวเลขของภาพเดิมจึงอาจต่างจาก 94.5 % เล็กน้อย
- `NOISE = protocol`: ตำแหน่งกล้อง + N(0, σ_t²) ต่อแกน, หมุนด้วย axis-angle ที่มุมต่อแกน ~ N(0, σ_r²) = error model ของ robustness protocol (แกน x ของรูปจึงเทียบกับรูป benchmark ได้ตรง ๆ);
  `fixed` = ระยะ/มุมคงที่ในทิศสุ่ม (แบบรูปเดิม)
- หนึ่ง draw ต่อภาพต่อระดับ: ระดับหนึ่ง ๆ จึงเป็นผลรวมของ draw อิสระเท่าจำนวนภาพ (~850); แถบในรูป = ควอร์ไทล์ระหว่างภาพ
- seed ของแต่ละ draw ขึ้นกับ (ชื่อภาพ, ชนิด error, ระดับ, ลำดับ draw) เท่านั้น → รันซ้ำ/รันต่อ/เลือกภาพบางส่วน ได้ตัวเลขเดิม
- จบแล้ว **Disconnect and delete runtime**